Generate 100,000+ records.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS identity;
CREATE SCHEMA IF NOT EXISTS identity.default;

In [0]:
%python
%pip install faker
from faker import Faker

import pandas as pd
import random

fake = Faker()

data = []

for i in range(100000):
    data.append({
        "username": fake.user_name(),
        "department": random.choice(
            ["HR","Finance","IT","Sales"]
        ),
        "login_status": random.choice(
            ["Success","Failed"]
        ),
        "risk_score": random.randint(1,100)
    })

df = pd.DataFrame(data)

In [0]:
%python
# Add device_type and login_time columns
import random
from datetime import datetime, timedelta

base_time = datetime(2026, 6, 1, 0, 0, 0)

df['device_type'] = [random.choice(["Desktop","Mobile","Tablet","Web"]) for _ in range(len(df))]
df['login_time'] = [base_time + timedelta(days=random.randint(0, 30), hours=random.randint(0, 23), minutes=random.randint(0, 59)) for _ in range(len(df))]

Bronze Layer

In [0]:
%python
# Convert datetime to string for Spark compatibility
df['login_time'] = df['login_time'].astype(str)
spark_df = spark.createDataFrame(df)

spark_df.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("identity.default.bronze_iam_logs")

Silver Layer

Clean data

In [0]:
%python
silver = (
 spark.table("identity.default.bronze_iam_logs")
 .dropDuplicates()
)

silver.write.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("identity.default.silver_iam_logs")

Gold

In [0]:
%python
from pyspark.sql.functions import count, avg

gold_df = silver.filter("login_status = 'Failed'") \
  .groupBy("department") \
  .agg(
    count("username").alias("failed_login_count"),
    avg("risk_score").alias("avg_risk_score")
  )

gold_df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("identity.default.gold_failed_login_summary")

Gold Layer

KPI 1: Failed Login Rate

In [0]:
SELECT
department,
COUNT(*) failures
FROM identity.default.silver_iam_logs
WHERE login_status='Failed'
GROUP BY department

KPI 2: High-Risk Users

In [0]:
SELECT
username,
AVG(risk_score)
FROM identity.default.silver_iam_logs
GROUP BY username

KPI 3: MFA Adoption

In [0]:
-- MFA data not available - showing login status distribution instead
SELECT
login_status,
COUNT(*) as count
FROM identity.default.silver_iam_logs
GROUP BY login_status

KPI 4: Privileged Account Usage

In [0]:
-- Role data not available - showing department usage instead
SELECT
department,
COUNT(*) as login_count
FROM identity.default.silver_iam_logs
GROUP BY department